# DL 파트 발표용 시각화 요약

탐지 모델(YOLO11) 학습·평가 결과가 여러 런 디렉터리와 개별 분석 노트북에 흩어져 있어,
발표에 바로 쓸 수 있게 핵심만 한 곳에 모았다. 무거운 추론은 다시 돌리지 않고 **이미
계산된 결과만 재사용**한다 (런 디렉터리의 PNG, 기존 리뷰 노트북에 이미 렌더링된 출력).

- 원본: `docs/STATUS.md`, `docs/runpod_distillation_20260831.md`,
  `notebooks/c5_own_night_review.ipynb`, `notebooks/c4f_distill_test_real_viz.ipynb`
- 이 노트북이 참조하는 이미지는 전부 `outputs/presentation/`에 모여 있고, 그 폴더만
  `.gitignore` 예외로 git에 함께 커밋된다 — `outputs/detect/` 등 나머지 산출물처럼
  로컬 재생성 없이도 클론 직후 바로 보인다.
- 원본을 갱신했다면 아래 첫 코드 셀로 `outputs/presentation/`을 다시 채울 수 있다
  (재생성 비용은 가벼움: 노트북 재실행 없이 JSON 디코드 + CSV 플로팅 + 파일 복사뿐).


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "scripts"))

from build_presentation_assets import main as build_presentation_assets

build_presentation_assets()  # outputs/presentation/ 전체를 다시 채움 (기존 노트북 출력 재사용 + CSV 플로팅 + 복사)
print("자료 준비 완료")


## 1. 주요 탐지 모델 학습곡선

| 모델 | 설명 |
|---|---|
| `c4b_loli0` | **현재 배포 중** · 2클래스(`person`·`bollard`) |
| `c4e_s3_11n` | 자체 학습 3클래스(`person`·`stairs`·`bollard`) **최우수 후보** · `C5` 자체 야간 27장 평가에서 우세 |

### `c4b_loli0` (배포 모델)
![c4b_loli0 results](../outputs/presentation/curve_c4b_loli0_results.png)

### `c4e_s3_11n` (최우수 후보)
![c4e_s3_11n results](../outputs/presentation/curve_c4e_s3_11n_results.png)

confusion matrix · PR curve:

![c4e_s3_11n confusion matrix](../outputs/presentation/curve_c4e_s3_11n_confusion_matrix.png)
![c4e_s3_11n PR curve](../outputs/presentation/curve_c4e_s3_11n_prcurve.png)


## 2. 지식 증류(Knowledge Distillation) 학습곡선

RunPod A100에서 교사(`c4f_11s_640_teacher`, YOLO11s) → 학생(`c4f_distill_11n_640`,
YOLO11n) 증류 학습을 진행했다(`docs/runpod_distillation_20260831.md` 5-1). 학습이
RunPod에서 이뤄져 Ultralytics가 로컬에 `results.png`를 남기지 않았다 — 위 코드 셀에서
`results.csv`로부터 즉석에서 재생성했다. 학생 쪽 `train/dis_loss`·`val/dis_loss`는
증류(teacher-student) 손실이다.

### 교사 `c4f_11s_640_teacher` (YOLO11s, 95 epoch)
![teacher results](../outputs/presentation/curve_teacher_results.png)

### 학생 `c4f_distill_11n_640` (YOLO11n, distilled, 100 epoch)
![student results](../outputs/presentation/curve_student_results.png)


## 3. 증류 평가 — 표본 크기가 결론을 뒤집은 사례

같은 5개 후보(교사 · 학생 FP32 · 학생 INT8 · `c4e_s3_11n` FP32 · `c4e_s3_11n` INT8)를
서로 다른 두 평가셋에서 쟀다(`docs/runpod_distillation_20260831.md` 5-3 · 5-6).

### 3-1. `own_night` (자체 촬영 27장 · GT person 4·stairs 6·bollard 13, conf 0.25)

| | mAP50(종합) | recall | precision | F1 | 음성오탐(전체) |
|---|---|---|---|---|---|
| 교사 `c4f_11s_640_teacher` | 0.600 | 0.652 | 0.625 | 0.638 | 2 |
| 학생 `c4f_distill_11n_640` | 0.554 | 0.652 | 0.556 | 0.600 | 5 |
| `c4e_s3_11n` (FP32) | 0.523 | 0.696 | 0.593 | 0.640 | 4 |
| `c4e_s3_11n` INT8 | 0.525 | 0.739 | 0.654 | 0.694 | 4 |

→ 언뜻 **교사가 우세**해 보인다.

### 3-2. NightOwls `rec34` held-out (1,001장 · pedestrian GT 1,576개, 학습 미사용)

| | mAP50 | mAP50-95 | precision | recall | stairs 오탐 |
|---|---|---|---|---|---|
| 교사 `c4f_11s_640_teacher` | 0.689 | 0.360 | 0.824 | 0.633 | 0.0% |
| **학생 `c4f_distill_11n_640` (FP32)** | **0.717** | 0.361 | 0.836 | **0.668** | 0.0% |
| 학생 INT8 | 0.692 | 0.362 | 0.850 | 0.631 | 0.0% |
| `c4e_s3_11n` (FP32) | 0.710 | 0.354 | 0.787 | 0.635 | 0.0% |
| `c4e_s3_11n` INT8 | 0.649 | 0.332 | 0.794 | 0.582 | 0.0% |

→ **방향이 뒤집힌다.** 학생이 5개 후보 중 1위, 교사는 오히려 최하위권.

**표본이 작은 own_night(GT 23개)만 보면 "교사가 낫다"로 오판하기 쉽지만, 표본이 훨씬 큰
rec34(GT 1,576개)에서는 정반대다.** `docs/STATUS.md` 3장 함정 1("판정은 항상 held-out에서,
한쪽만 보고 결론 내리지 말 것")이 실제로 들어맞은 사례 — 결론은 "학생이 무조건 낫다"가
아니라 "표본이 큰 쪽을 더 신뢰하되 원인은 아직 미상"이다. 양자화(INT8) 효과만은 두
도메인에서 방향이 일치한다(학생·`c4e_s3_11n` 모두 INT8이 FP32보다 낮음).

### PR curve — `rec34` 교사 vs 학생

| 교사 `c4f_11s_640_teacher` | 학생 `c4f_distill_11n_640` |
|---|---|
| ![teacher PR curve rec34](../outputs/presentation/curve_teacher_rec34_prcurve.png) | ![student PR curve rec34](../outputs/presentation/curve_student_rec34_prcurve.png) |

> **읽는 법 — 왜 곡선이 recall 0.7~0.8 부근에서 precision 0으로 수직 낙하하는가.**
> 버그가 아니라 Ultralytics가 AP를 COCO/VOC 방식으로 계산하는 정의에서 나오는 그림이다.
> `ap_per_class`/`compute_ap`(`ultralytics/utils/metrics.py:758`)는 AP를 항상
> `recall ∈ [0, 1]` 전 구간에 대해 적분하도록, 실제 곡선의 끝(모델이 confidence를 0까지
> 낮춰도 도달하는 최대 recall)에 sentinel 점 `(recall[-1], precision=0)`과 `(1.0, 0)`을
> 추가한다. 그래서 그 최대 recall 지점에서 precision이 수직으로 0까지 떨어지고 1.0까지
> 평평하게 이어진다 — 이 프로젝트의 `BoxPR_curve.png` 전부(예: 위 `c4e_s3_11n`도 동일)에서
> 공통으로 나타나는 패턴이다. 실제 의미는 "confidence를 아무리 낮춰도 이 모델은 GT의
> 일부(교사는 약 25%)를 완전히 못 잡는다"는 **recall 상한**이 존재한다는 것 — 그 상한이
> 어디서 꺾이는지(위 그래프에서 교사 쪽이 더 일찍 꺾인다)가 두 모델을 비교할 때 보는
> 지점이다.


## 4. 후보 모델 4종 비교 — 자체 야간 27장 중 대표 5장

`c4b_loli0`(배포) · `c4d_11n_640`(외부 기여) · `c4e_s3_11n`(자체 최우수) ·
`c4e_s3_11n`-INT8 네 모델을 같은 이미지에 나란히 예측시킨 것이다(전체 27장 비교는
`notebooks/c5_own_night_review.ipynb`).

![bollard 비교](../outputs/presentation/01_bollard.png)
![person+bollard+횡단보도 비교](../outputs/presentation/02_person_bollard_crosswalk.png)
![stairs 비교](../outputs/presentation/03_stairs.png)
![person 3명 비교](../outputs/presentation/04_three_person.png)
![음성(오탐 확인) 비교](../outputs/presentation/05_negative_fp.png)

### 정량 비교 (자체 야간 27장)

mAP50:

| | person | stairs | bollard | 종합 |
|---|---|---|---|---|
| c4b_loli0 | 0.495 | 0.089 | 0.000 | 0.195 |
| c4d_11n_640 | 0.995 | 0.051 | 0.548 | 0.531 |
| c4e_s3_11n | 0.995 | 0.090 | 0.568 | 0.551 |
| c4e_s3_11n_INT8 | 0.995 | 0.074 | 0.510 | 0.526 |

음성 프레임 오탐:

| | 음성프레임 | 오탐박스 | 발화프레임 |
|---|---|---|---|
| c4b_loli0 | 27 | 13 | 11 |
| c4d_11n_640 | 27 | 11 | 9 |
| c4e_s3_11n | 27 | 4 | 4 |
| c4e_s3_11n_INT8 | 27 | 4 | 4 |


## 5. 증류 학생 모델 단독 확인 — `test_real_data` 7장

`c4f_distill_11n_640`(학생)의 예측과 GT 라벨을 나란히 확인한다. ⚠️ 라벨이 있는 정량
확인이 아니라 **눈으로 보는 정성 확인**이다(표본 7장 · recall/FP 정량 근거로 쓰지 말 것).

### 예측
![student predictions](../outputs/presentation/06_distill_predictions.png)

### GT 라벨
![student GT labels](../outputs/presentation/07_distill_gt.png)


## 결론 · 남은 것

- 지식 증류 실행은 완료했지만(교사→학생, ONNX, INT8) **아직 배포 후보로 채택하지 않았다**
  — `docs/distillation_plan_20260829.md` 5-6절 채택 기준(`C11` 속도 + `C5` 야간 실측)을
  통과해야 한다.
- own_night과 rec34가 교사/학생 우열을 뒤집는 원인은 **아직 규명되지 않았다** — `C2` 자체
  촬영이 늘어야 own_night 표본 신뢰도가 개선된다.
- 상세 절차·원문 수치는 `docs/runpod_distillation_20260831.md` 5장 참고.
